# Step 1: Install required packages


In [4]:
%pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2: Import dependencies

In [5]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from dotenv import load_dotenv

C:\Users\edmar\AppData\Local\Temp\ipykernel_17560\172370802.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


# Step 3: Set OpenAI credentials (note I am uning my own key)

In [6]:
# Load environment variables from .env file
load_dotenv()

True

# Step 4: Load and chunk the directory of PDF documents

In [13]:
# Load and chunk pdf documents

loader = DirectoryLoader("./MondayDocumentation", glob="**/*.pdf", loader_cls=PyPDFLoader)
documents = loader.load()

# Note that each page in the document is treated as a separate document. This is because the PyPDFLoader splits the PDF into pages by default.
print(f"Number of pdf document: {len(documents)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)
if len(chunks) == 0:
    raise ValueError("No chunks were created. Please check the document loading and splitting process.")

average_chunk_length = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)

print(f"Created {len(chunks)} chunks")

print(f"Average chunk length: {average_chunk_length:.2f} characters")






Number of pdf document: 39
Created 80 chunks
Average chunk length: 334.56 characters


# Step 5: Create the vector store using OPenAI Embeddings

In [ ]:
embeddings = OpenAIEmbeddings(
   
    api_version="2023-05-15",
    deployment="text-embedding-ada-002",
    api_key=os.environ["OPENAI_API_KEY"]
)

vectorstore = FAISS.from_documents(chunks, embeddings)
# this will save the vectorstore to a local directory called "faiss_index". You can load it later using FAISS.load_local("faiss_index", embeddings).
# this will be cheaper to load the vectorstore in the future without having to reprocess the documents and compute the embeddings again.

vectorstore.save_local("faiss_index")


# Step 6: Initialise the Azure OpenAI LLM

In [9]:
# Initialize the gpt-5-mini model from Azure with temperature 0 for deterministic output

llm = ChatOpenAI(    
    model="gpt-5-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

# Step 7: Create the RAG Chain

In [10]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Step 8: Ask a question

In [11]:
query = "How do I get a list of columns in a board using the monday.com API?"
result = qa_chain.invoke({"query": query})

# Step 9 Print Results

In [14]:
print("Question:", query)
print("\nAnswer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

Question: How do I get a list of columns in a board using the monday.com API?

Answer: Use the boards -> columns GraphQL field. Columns cannot be queried at the root — you must request them from a board. The request requires the boards:read scope.

Example GraphQL query (get id, title, type, settings_str):
query {
  boards(ids: [1234567890]) {
    columns {
      id
      title
      type
      settings_str
    }
  }
}

Example curl call:
curl -X POST https://api.monday.com/v2 \
  -H "Content-Type: application/json" \
  -H "Authorization: YOUR_API_TOKEN" \
  -d '{"query":"query { boards(ids: [1234567890]) { columns { id title type settings_str } } }"}'

Example JavaScript (fetch):
fetch('https://api.monday.com/v2', {
  method: 'POST',
  headers: {
    'Content-Type': 'application/json',
    'Authorization': 'YOUR_API_TOKEN'
  },
  body: JSON.stringify({
    query: `query {
      boards(ids: [1234567890]) {
        columns {
          id
          title
          type
          settings